# Assignment 6 - Myanmar and Hieroglyphic Translation using Marian NMT 

## Objective
This project implements Neural Machine Translation between Myanmar and Hieroglyphic text using the Marian NMT framework. 

The experiments and workflow include:

1. Data preprocessing (Syllable segmentation using sylbreak4all)
2. Train / validation / test splitting
3. Vocabulary construction (Syllable unit)
4. Sequence-to-Sequence (RNN) NMT
5. Transformer NMT
6. Myanmar → Hieroglyphic translation
7. Hieroglyphic → Myanmar translation
8. Hyperparameter tuning
9. Model evaluation using BLEU score
10. Result Summary (Model Comparison)

# Building Environment

In [2]:
import os


os.makedirs(os.path.expanduser("~/assignment-6/myhi_data"), exist_ok=True)


os.environ["PATH"] = "/home/thida/marian-dev/build:" + os.environ["PATH"]

print("Environment configured successfully!")

Environment configured successfully!


# Preparing Data

In [3]:
import os


data_dir = os.path.expanduser("~/assignment-6/myhi_data")
os.chdir(data_dir)

!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/train.my
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/train.hi
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/valid.my
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/valid.hi
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.my
!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/AIE-F-B2/main/assignment/assignment-6/myhi_data/test.hi

print("Downloaded files:")
!ls -l

File ‘train.my’ already there; not retrieving.

File ‘train.hi’ already there; not retrieving.

File ‘valid.my’ already there; not retrieving.

File ‘valid.hi’ already there; not retrieving.

File ‘test.my’ already there; not retrieving.

File ‘test.hi’ already there; not retrieving.

Downloaded files:
total 1068
drwxr-xr-x 2 thida thida   4096 Oct  9 13:16 syllable_segmented
-rw-r--r-- 1 thida thida  24117 Oct  9 13:12 test.hi
-rw-r--r-- 1 thida thida  29994 Oct  9 13:12 test.my
-rw-r--r-- 1 thida thida 412919 Oct  9 13:12 train.hi
-rw-r--r-- 1 thida thida 502978 Oct  9 13:12 train.my
-rw-r--r-- 1 thida thida  48906 Oct  9 13:12 valid.hi
-rw-r--r-- 1 thida thida  57890 Oct  9 13:12 valid.my
drwxr-xr-x 2 thida thida   4096 Oct  9 13:18 vocab


# Syllable Segmentation

In [4]:
import os

os.chdir(os.path.expanduser("~/assignment-6"))
os.makedirs("sylbreak", exist_ok=True)
os.chdir("sylbreak")

!wget -nc https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak4all/refs/heads/main/python/sylbreak4all.py

os.chdir(os.path.expanduser("~/assignment-6/myhi_data"))
os.makedirs("syllable_segmented", exist_ok=True)

!python ~/assignment-6/sylbreak/sylbreak4all.py -i train.my -l bm -s " " > syllable_segmented/syl_train.my
!python ~/assignment-6/sylbreak/sylbreak4all.py -i valid.my -l bm -s " " > syllable_segmented/syl_valid.my
!python ~/assignment-6/sylbreak/sylbreak4all.py -i test.my -l bm -s " " > syllable_segmented/syl_test.my

print("Syllable segmentation completed successfully!")
print("Sample segmented training lines:")
!head -n 5 syllable_segmented/syl_train.my

File ‘sylbreak4all.py’ already there; not retrieving.

Syllable segmentation completed successfully!
Sample segmented training lines:
မင်း က အေး ဆေး တယ်
ရိုး သား မှု ရှိ တယ်
ကျွန် မ ရိုး သား နိုင် လား ။
ဘယ် လို သက် သာ မှု လဲ ။
ကျွန် မ သွား ခွ င့် ပြု ပါ ။


In [5]:
!time python /home/thida/assignment-6/sylbreak/sylbreak4all.py -i /home/thida/assignment-6/myhi_data/train.my -l bm -s " " > /home/thida/assignment-6/myhi_data/syllable_segmented/syl_train.my

!time python /home/thida/assignment-6/sylbreak/sylbreak4all.py -i /home/thida/assignment-6/myhi_data/test.my -l bm -s " " > /home/thida/assignment-6/myhi_data/syllable_segmented/syl_test.my

!time python /home/thida/assignment-6/sylbreak/sylbreak4all.py -i /home/thida/assignment-6/myhi_data/valid.my -l bm -s " " > /home/thida/assignment-6/myhi_data/syllable_segmented/syl_valid.my

print("Segmentation completed successfully!")


real	0m0.077s
user	0m0.073s
sys	0m0.004s

real	0m0.038s
user	0m0.024s
sys	0m0.008s

real	0m0.028s
user	0m0.028s
sys	0m0.000s
Segmentation completed successfully!


In [6]:
!head -n 10 /home/thida/assignment-6/myhi_data/syllable_segmented/syl_test.my

အ ချစ် က နာ ကျင် စေ တယ် ။
ကျွန် မ မှာ အ စား အ စာ မ ရှိ ဘူး ။
ဘယ် သူ မှ ကျွန် မ ကို မ မြင် ခဲ့ ဘူး ။
ခင် ဗျား က သ တိ ထား တတ် သူ ပါ ။
အ သင်း က စော င့် နေ ခဲ့ တယ် ။
ခ ရီး ကောင်း ပါ
ဘယ် လို ပဲ ဖြစ် ဖြစ် လုပ် ခဲ့ တယ် ။
အ ခန်း က မှောင် နေ တယ် ။
သူ မ လမ်း လျှောက် တယ် ။
ဒါ ပဲ ဆို


In [7]:
!head -n 10 /home/thida/assignment-6/myhi_data/syllable_segmented/syl_valid.my

ကျွန် မ အ ဆင် ပြေ ပါ တယ် ။
ကျွန် မ မှာ မြင်း မ ရှိ ဘူး ။
သူ့ ကို ငို စေ ခဲ့ တယ် ။
နည်း နည်း ပို နီး လာ ပါ ဦး ။
ကျွန် တော် ယုံ ကြည် ပါ တယ် ။
ချို တာ ကြိုက် တယ် ။
စိတ် မ ကောင်း ပါ ဘူး ။
ဒါ က အ ရ သာ ကောင်း တယ် ။
ပို ကောင်း လာ ပြီ ။
ကျွန် မ က ယာ ဗီ ယာ စား ခဲ့ တယ် ။


In [8]:
!head -n 10 /home/thida/assignment-6/myhi_data/syllable_segmented/syl_train.my

မင်း က အေး ဆေး တယ်
ရိုး သား မှု ရှိ တယ်
ကျွန် မ ရိုး သား နိုင် လား ။
ဘယ် လို သက် သာ မှု လဲ ။
ကျွန် မ သွား ခွ င့် ပြု ပါ ။
ကျွန် တော် တို့ မြင် ခဲ့ တယ် ။
ကို ယ့် ကိုယ် ကို လေး စား ပါ ။
သ န့် ရှင်း လား ။
ငါး တစ် ကောင် က ရေ ကူး နိုင် တယ်
ဆေး လိပ် သောက် တာ ရပ် လိုက် ပါ


In [10]:
!wc -l ~/assignment-6/myhi_data/train.my ~/assignment-6/myhi_data/train.hi

  8500 /home/thida/assignment-6/myhi_data/train.my
  8500 /home/thida/assignment-6/myhi_data/train.hi
 17000 total


In [11]:
!wc -l ~/assignment-6/myhi_data/valid.my ~/assignment-6/myhi_data/valid.hi ~/assignment-6/myhi_data/test.my ~/assignment-6/myhi_data/test.hi

  1000 /home/thida/assignment-6/myhi_data/valid.my
  1000 /home/thida/assignment-6/myhi_data/valid.hi
   500 /home/thida/assignment-6/myhi_data/test.my
   500 /home/thida/assignment-6/myhi_data/test.hi
  3000 total


In [12]:
!head -n 5 ~/assignment-6/myhi_data/train.my
!head -n 5 ~/assignment-6/myhi_data/train.hi

မင်းက အေးဆေးတယ်
ရိုးသားမှုရှိတယ်
ကျွန်မ ရိုးသားနိုင်လား။
ဘယ်လို သက်သာမှုလဲ။
ကျွန်မသွားခွင့်ပြုပါ။
𓂨𓀀𓎡𓋴𓂝 𓏮𓀀𓐍𓂋
𓏏𓂝𓋴𓂝 𓈙𓆑𓎛𓏭𓂝𓋴𓂝
𓏭𓄿𓎛 𓆑 𓇋𓂝 𓊪𓐍𓎛𓂝𓈙𓈎
𓏏𓊪𓄿𓈎 𓄿 𓋴𓂝𓂋𓆑𓂝𓅱
𓄿𓂋𓂋𓐍𓏏 𓉔𓂝 𓈎𓐍 𓃀𓐍


# Creating Vocab 

In [9]:
!marian --help | head -n 25

Marian: Fast Neural Machine Translation in C++
Usage: marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=2048               Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zone TEXT  

In [13]:
import os
os.makedirs(os.path.expanduser("~/assignment-6/myhi_data/vocab"), exist_ok=True)

!cat ~/assignment-6/myhi_data/syllable_segmented/syl_test.my \
     ~/assignment-6/myhi_data/test.hi \
     ~/assignment-6/myhi_data/syllable_segmented/syl_train.my \
     ~/assignment-6/myhi_data/train.hi \
     ~/assignment-6/myhi_data/syllable_segmented/syl_valid.my \
     ~/assignment-6/myhi_data/valid.hi \
     | marian-vocab > ~/assignment-6/myhi_data/vocab/vocab.syl.yml

print("Vocab created successfully!")

[2026-10-10 01:25:59] Creating vocabulary...
[2026-10-10 01:25:59] [data] Creating vocabulary stdout from stdin
[2026-10-10 01:25:59] Finished
Vocab created successfully!


In [14]:
!head -n 15 ~/assignment-6/myhi_data/vocab/vocab.syl.yml

</s>: 0
<unk>: 1
။: 2
တယ်: 3
မ: 4
ပါ: 5
က: 6
𓆑: 7
ကျွန်: 8
ကို: 9
အ: 10
သူ: 11
ခဲ့: 12
နေ: 13
ဒါ: 14


In [15]:
import os

root_dir = os.path.expanduser("~/assignment-6")
print(f"Directory structure for: {root_dir}\n")

for root, dirs, files in os.walk(root_dir):
    level = root.replace(root_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root) or root}/")
    sub_indent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{sub_indent}{f}")

Directory structure for: /home/thida/assignment-6

assignment-6/
    s2s.syl.sh
    data/
    myhi_data/
        test.my
        train.my
        test.hi
        valid.my
        train.hi
        valid.hi
        vocab/
            vocab.syl.yml
        syllable_segmented/
            syl_valid.my
            syl_test.my
            syl_train.my
    sylbreak/
        sylbreak4all.py


In [16]:
%cd ~/assignment-6/

/home/thida/assignment-6


# Script for Sequence to Sequence Model

In [21]:
%%writefile s2s.syl.sh
#!/bin/bash

model_folder="model.s2s.syl";
mkdir -p ${model_folder};
base_path="/home/thida/assignment-6/myhi_data";

# Marian binary path
MARIAN_BIN="/home/thida/marian-dev/build/marian"

${MARIAN_BIN} \
  --type s2s \
  --train-sets ${base_path}/syllable_segmented/syl_train.my ${base_path}/train.hi \
  --max-length 200 \
  --valid-sets ${base_path}/syllable_segmented/syl_valid.my ${base_path}/valid.hi \
  --vocabs  ${base_path}/vocab/vocab.syl.yml  ${base_path}/vocab/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --workspace 500 \
  --enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
  --dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
  --tied-embeddings --layer-normalization --skip \
  --mini-batch-fit \
  --valid-mini-batch 32 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-freq 1000 --save-freq 1000 --disp-freq 500 \
  --dropout-rnn 0.3 --dropout-src 0.3 --exponential-smoothing \
  --early-stopping 10 \
  --log ${model_folder}/train.log --valid-log ${model_folder}/valid.log \
  --devices 0 --sync-sgd --seed 1111  \
  --dump-config > ${model_folder}/my-hi.config.yml

time ${MARIAN_BIN} -c ${model_folder}/my-hi.config.yml  2>&1 | tee ${model_folder}/s2s.my-hi.log

Overwriting s2s.syl.sh


In [22]:
!time bash s2s.syl.sh

[2026-10-10 01:34:23] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 01:34:23] [marian] Running on Thida as process 3944 with command line:
[2026-10-10 01:34:23] [marian] /home/thida/marian-dev/build/marian -c model.s2s.syl/my-hi.config.yml
[2026-10-10 01:34:23] [config] after: 0e
[2026-10-10 01:34:23] [config] after-batches: 0
[2026-10-10 01:34:23] [config] after-epochs: 0
[2026-10-10 01:34:23] [config] all-caps-every: 0
[2026-10-10 01:34:23] [config] allow-unk: false
[2026-10-10 01:34:23] [config] authors: false
[2026-10-10 01:34:23] [config] beam-size: 12
[2026-10-10 01:34:23] [config] bert-class-symbol: "[CLS]"
[2026-10-10 01:34:23] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 01:34:23] [config] bert-masking-fraction: 0.15
[2026-10-10 01:34:23] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 01:34:23] [config] bert-train-type-embeddings: true
[2026-10-10 01:34:23] [config] bert-type-vocab-size: 2
[2026-10-10 01:34:23] [config] build-info: ""
[2026-10-10 01

In [24]:
rm -rf model.s2s.syl

update the script

In [ ]:
%%writefile s2s.syl.sh
#!/bin/bash

model_folder="model.s2s.syl"
mkdir -p ${model_folder}
base_path="/home/thida/assignment-6/myhi_data"

MARIAN_BIN="/home/thida/marian-dev/build/marian"

${MARIAN_BIN} \
  --type s2s \
  --train-sets ${base_path}/syllable_segmented/syl_train.my ${base_path}/train.hi \
  --max-length 50 \
  --valid-sets ${base_path}/syllable_segmented/syl_valid.my ${base_path}/valid.hi \
  --vocabs ${base_path}/vocab/vocab.syl.yml ${base_path}/vocab/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --dim-emb 128 \
  --dim-rnn 128 \
  --enc-type bidirectional \
  --enc-cell gru \
  --enc-depth 1 \
  --dec-cell gru \
  --dec-depth 1 \
  --dropout-rnn 0.1 \
  --dropout-src 0.1 \
  --dropout-trg 0.1 \
  --mini-batch 64 \
  --learn-rate 0.0005 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  --log ${model_folder}/train.log \
  --valid-log ${model_folder}/valid.log \
  --devices 0 \
  --overwrite \
  --dump-config > ${model_folder}/my-hi.config.yml

time ${MARIAN_BIN} -c ${model_folder}/my-hi.config.yml 2>&1 | tee ${model_folder}/s2s.my-hi.log

Overwriting s2s.syl.sh


train model from Myanmar to Hiero

In [26]:
!bash s2s.syl.sh

[2026-10-10 09:48:12] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 09:48:12] [marian] Running on Thida as process 9874 with command line:
[2026-10-10 09:48:12] [marian] /home/thida/marian-dev/build/marian -c model.s2s.syl/my-hi.config.yml
[2026-10-10 09:48:12] [config] after: 0e
[2026-10-10 09:48:12] [config] after-batches: 0
[2026-10-10 09:48:12] [config] after-epochs: 0
[2026-10-10 09:48:12] [config] all-caps-every: 0
[2026-10-10 09:48:12] [config] allow-unk: false
[2026-10-10 09:48:12] [config] authors: false
[2026-10-10 09:48:12] [config] beam-size: 12
[2026-10-10 09:48:12] [config] bert-class-symbol: "[CLS]"
[2026-10-10 09:48:12] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 09:48:12] [config] bert-masking-fraction: 0.15
[2026-10-10 09:48:12] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 09:48:12] [config] bert-train-type-embeddings: true
[2026-10-10 09:48:12] [config] bert-type-vocab-size: 2
[2026-10-10 09:48:12] [config] build-info: ""
[2026-10-10 09

Testing the model

In [27]:
%%writefile evaluate.sh
#!/bin/bash


DATA="/home/thida/assignment-6/myhi_data"
MODEL_DIR="model.s2s.syl"
MARIAN_DEC="/home/thida/marian-dev/build/marian-decoder"

VOCAB="${DATA}/vocab/vocab.syl.yml"
TEST_SRC="${DATA}/syllable_segmented/syl_test.my"
TEST_TGT="${DATA}/test.hi"

> eval-result.txt

MODEL="${MODEL_DIR}/model.npz.best-bleu.npz"
HYP="${MODEL_DIR}/hyp.best-bleu.hi"

${MARIAN_DEC} \
    -m "${MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    --devices 0 \
    --output "${HYP}" \
    < "${TEST_SRC}"

echo "Evaluation with ${HYP}, Best BLEU model:" >> eval-result.txt
cat "${HYP}" | sacrebleu "${TEST_TGT}" >> eval-result.txt

MODEL="${MODEL_DIR}/model.npz.best-ce-mean-words.npz"
HYP="${MODEL_DIR}/hyp.best-ce-mean-words.hi"

${MARIAN_DEC} \
    -m "${MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    --devices 0 \
    --output "${HYP}" \
    < "${TEST_SRC}"

echo "Evaluation with ${HYP}, Best CE model:" >> eval-result.txt
cat "${HYP}" | sacrebleu "${TEST_TGT}" >> eval-result.txt

MODEL="${MODEL_DIR}/model.npz"
HYP="${MODEL_DIR}/hyp.final.hi"

${MARIAN_DEC} \
    -m "${MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    --devices 0 \
    --output "${HYP}" \
    < "${TEST_SRC}"

echo "Evaluation with ${HYP}, Final model:" >> eval-result.txt
cat "${HYP}" | sacrebleu "${TEST_TGT}" >> eval-result.txt

echo "Evaluation Finished."

Writing evaluate.sh


In [29]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 178.1 kB/s eta 0:00:0000:0200:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 215.8 kB/s eta 0:00:00a 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.2/807.2 kB 120.0 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sacrebleu]/7 [sacrebleu]


In [30]:
!bash evaluate.sh

[2026-10-10 10:25:18] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 10:25:18] [marian] Running on Thida as process 10479 with command line:
[2026-10-10 10:25:18] [marian] /home/thida/marian-dev/build/marian-decoder -m model.s2s.syl/model.npz.best-bleu.npz -v /home/thida/assignment-6/myhi_data/vocab/vocab.syl.yml /home/thida/assignment-6/myhi_data/vocab/vocab.syl.yml --devices 0 --output model.s2s.syl/hyp.best-bleu.hi
[2026-10-10 10:25:18] [config] alignment: ""
[2026-10-10 10:25:18] [config] allow-special: false
[2026-10-10 10:25:18] [config] allow-unk: false
[2026-10-10 10:25:18] [config] authors: false
[2026-10-10 10:25:18] [config] beam-size: 12
[2026-10-10 10:25:18] [config] bert-class-symbol: "[CLS]"
[2026-10-10 10:25:18] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 10:25:18] [config] bert-masking-fraction: 0.15
[2026-10-10 10:25:18] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 10:25:18] [config] bert-train-type-embeddings: true
[2026-10-10 10:25:18] 

In [31]:
!cat eval-result.txt

Evaluation with model.s2s.syl/hyp.best-bleu.hi, Best BLEU model:
{
 "name": "BLEU",
 "score": 29.5,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "54.7/32.3/23.4/22.0 (BP = 0.955 ratio = 0.956 hyp_len = 1504 ref_len = 1573)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}
Evaluation with model.s2s.syl/hyp.best-ce-mean-words.hi, Best CE model:
{
 "name": "BLEU",
 "score": 29.5,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "54.7/32.3/23.4/22.0 (BP = 0.955 ratio = 0.956 hyp_len = 1504 ref_len = 1573)",
 "nrefs": "1",
 "case": "mixed",
 "eff": "no",
 "tok": "13a",
 "smooth": "exp",
 "version": "2.6.0"
}
Evaluation with model.s2s.syl/hyp.final.hi, Final model:
{
 "name": "BLEU",
 "score": 26.4,
 "signature": "nrefs:1|case:mixed|eff:no|tok:13a|smooth:exp|version:2.6.0",
 "verbose_score": "52.5/29.2/19.2/17.5 (BP = 0.986 ratio = 0.986 hyp_len 

Train Herio-to myanamr using seq2seq

In [ ]:
%%writefile s2s.hi-my.sh
#!/bin/bash

model_folder="model.hi-my"
mkdir -p ${model_folder}
base_path="/home/thida/assignment-6/myhi_data"

MARIAN_BIN="/home/thida/marian-dev/build/marian"

${MARIAN_BIN} \
  --type s2s \
  --train-sets ${base_path}/train.hi ${base_path}/syllable_segmented/syl_train.my \
  --max-length 50 \
  --valid-sets ${base_path}/valid.hi ${base_path}/syllable_segmented/syl_valid.my \
  --vocabs ${base_path}/vocab/vocab.syl.yml ${base_path}/vocab/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --dim-emb 128 \
  --dim-rnn 128 \
  --enc-type bidirectional \
  --enc-cell gru \
  --enc-depth 1 \
  --dec-cell gru \
  --dec-depth 1 \
  --dropout-rnn 0.1 \
  --dropout-src 0.1 \
  --dropout-trg 0.1 \
  --mini-batch 64 \
  --learn-rate 0.0005 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  --log ${model_folder}/train.log \
  --valid-log ${model_folder}/valid.log \
  --devices 0 \
  --overwrite \
  --dump-config > ${model_folder}/hi-my.config.yml

time ${MARIAN_BIN} -c ${model_folder}/hi-my.config.yml 2>&1 | tee ${model_folder}/s2s.hi-my.log

Writing s2s.hi-my.sh


In [33]:
!bash s2s.hi-my.sh

[2026-10-10 10:34:37] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 10:34:37] [marian] Running on Thida as process 10670 with command line:
[2026-10-10 10:34:37] [marian] /home/thida/marian-dev/build/marian -c model.hi-my/hi-my.config.yml
[2026-10-10 10:34:37] [config] after: 0e
[2026-10-10 10:34:37] [config] after-batches: 0
[2026-10-10 10:34:37] [config] after-epochs: 0
[2026-10-10 10:34:37] [config] all-caps-every: 0
[2026-10-10 10:34:37] [config] allow-unk: false
[2026-10-10 10:34:37] [config] authors: false
[2026-10-10 10:34:37] [config] beam-size: 12
[2026-10-10 10:34:37] [config] bert-class-symbol: "[CLS]"
[2026-10-10 10:34:37] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 10:34:37] [config] bert-masking-fraction: 0.15
[2026-10-10 10:34:37] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 10:34:37] [config] bert-train-type-embeddings: true
[2026-10-10 10:34:37] [config] bert-type-vocab-size: 2
[2026-10-10 10:34:37] [config] build-info: ""
[2026-10-10 10:

Testing

In [34]:
%%writefile evaluate_hi_my.sh
#!/bin/bash

DATA="/home/thida/assignment-6/myhi_data"
MODEL_DIR="model.hi-my"
MARIAN_DEC="/home/thida/marian-dev/build/marian-decoder"

VOCAB="${DATA}/vocab/vocab.syl.yml"
TEST_SRC="${DATA}/test.hi"
TEST_TGT="${DATA}/syllable_segmented/syl_test.my"

> eval-result-hi-my.txt

# 1. Best BLEU model
MODEL="${MODEL_DIR}/model.npz.best-bleu.npz"
HYP="${MODEL_DIR}/hyp.best-bleu.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best BLEU model  : BLEU = ${SCORE}" >> eval-result-hi-my.txt

# 2. Best CE model
MODEL="${MODEL_DIR}/model.npz.best-ce-mean-words.npz"
HYP="${MODEL_DIR}/hyp.best-ce-mean-words.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best CE model    : BLEU = ${SCORE}" >> eval-result-hi-my.txt

# 3. Final model
MODEL="${MODEL_DIR}/model.npz"
HYP="${MODEL_DIR}/hyp.final.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Final model      : BLEU = ${SCORE}" >> eval-result-hi-my.txt

Writing evaluate_hi_my.sh


In [36]:
!bash evaluate_hi_my.sh

In [37]:
!cat eval-result-hi-my.txt

Best BLEU model  : BLEU = 47.0
Best CE model    : BLEU = 45.0
Final model      : BLEU = 47.0


training with Transformer model
Myanmar-to-Herio

In [38]:
%%writefile transformer.my-hi.sh
#!/bin/bash

model_folder="model.transformer.my-hi"
mkdir -p ${model_folder}
base_path="/home/thida/assignment-6/myhi_data"

MARIAN_BIN="/home/thida/marian-dev/build/marian"

${MARIAN_BIN} \
  --type transformer \
  --train-sets ${base_path}/syllable_segmented/syl_train.my ${base_path}/train.hi \
  --max-length 50 \
  --valid-sets ${base_path}/syllable_segmented/syl_valid.my ${base_path}/valid.hi \
  --vocabs ${base_path}/vocab/vocab.syl.yml ${base_path}/vocab/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --dim-emb 256 \
  --transformer-dim-ffn 1024 \
  --transformer-heads 4 \
  --enc-depth 4 \
  --dec-depth 4 \
  --mini-batch 64 \
  --optimizer adam \
  --learn-rate 0.0003 \
  --lr-warmup 4000 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  --log ${model_folder}/train.log \
  --valid-log ${model_folder}/valid.log \
  --devices 0 \
  --overwrite \
  --dump-config > ${model_folder}/transformer.my-hi.config.yml

time ${MARIAN_BIN} -c ${model_folder}/transformer.my-hi.config.yml 2>&1 | tee ${model_folder}/transformer.my-hi.log

Writing transformer.my-hi.sh


In [39]:
!bash transformer.my-hi.sh

[2026-10-10 11:13:43] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 11:13:43] [marian] Running on Thida as process 11415 with command line:
[2026-10-10 11:13:43] [marian] /home/thida/marian-dev/build/marian -c model.transformer.my-hi/transformer.my-hi.config.yml
[2026-10-10 11:13:43] [config] after: 0e
[2026-10-10 11:13:43] [config] after-batches: 0
[2026-10-10 11:13:43] [config] after-epochs: 0
[2026-10-10 11:13:43] [config] all-caps-every: 0
[2026-10-10 11:13:43] [config] allow-unk: false
[2026-10-10 11:13:43] [config] authors: false
[2026-10-10 11:13:43] [config] beam-size: 12
[2026-10-10 11:13:43] [config] bert-class-symbol: "[CLS]"
[2026-10-10 11:13:43] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 11:13:43] [config] bert-masking-fraction: 0.15
[2026-10-10 11:13:43] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 11:13:43] [config] bert-train-type-embeddings: true
[2026-10-10 11:13:43] [config] bert-type-vocab-size: 2
[2026-10-10 11:13:43] [config] build-

In [40]:
%%writefile evaluate_trans_my_hi.sh
#!/bin/bash

DATA="/home/thida/assignment-6/myhi_data"
MODEL_DIR="model.transformer.my-hi"
MARIAN_DEC="/home/thida/marian-dev/build/marian-decoder"

VOCAB="${DATA}/vocab/vocab.syl.yml"
TEST_SRC="${DATA}/syllable_segmented/syl_test.my"
TEST_TGT="${DATA}/test.hi"

> eval-result-trans-my-hi.txt

# 1. Best BLEU model
MODEL="${MODEL_DIR}/model.npz.best-bleu.npz"
HYP="${MODEL_DIR}/hyp.best-bleu.hi"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best BLEU model  : BLEU = ${SCORE}" >> eval-result-trans-my-hi.txt

# 2. Best CE model
MODEL="${MODEL_DIR}/model.npz.best-ce-mean-words.npz"
HYP="${MODEL_DIR}/hyp.best-ce-mean-words.hi"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best CE model    : BLEU = ${SCORE}" >> eval-result-trans-my-hi.txt

# 3. Final model
MODEL="${MODEL_DIR}/model.npz"
HYP="${MODEL_DIR}/hyp.final.hi"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Final model      : BLEU = ${SCORE}" >> eval-result-trans-my-hi.txt

Writing evaluate_trans_my_hi.sh


In [41]:
!bash evaluate_trans_my_hi.sh

In [42]:
!cat eval-result-trans-my-hi.txt

Best BLEU model  : BLEU = 22.6
Best CE model    : BLEU = 22.4
Final model      : BLEU = 22.6


In [ ]:
Training herio-to-myanmar

In [ ]:
%%writefile transformer.hi-my.sh
#!/bin/bash

model_folder="model.transformer.hi-my"
mkdir -p ${model_folder}
base_path="/home/thida/assignment-6/myhi_data"

MARIAN_BIN="/home/thida/marian-dev/build/marian"

${MARIAN_BIN} \
  --type transformer \
  --train-sets ${base_path}/train.hi ${base_path}/syllable_segmented/syl_train.my \
  --max-length 50 \
  --valid-sets ${base_path}/valid.hi ${base_path}/syllable_segmented/syl_valid.my \
  --vocabs ${base_path}/vocab/vocab.syl.yml ${base_path}/vocab/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --dim-emb 256 \
  --transformer-dim-ffn 1024 \
  --transformer-heads 4 \
  --enc-depth 4 \
  --dec-depth 4 \
  --mini-batch 64 \
  --optimizer adam \
  --learn-rate 0.0003 \
  --lr-warmup 4000 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  --log ${model_folder}/train.log \
  --valid-log ${model_folder}/valid.log \
  --devices 0 \
  --overwrite \
  --dump-config > ${model_folder}/transformer.hi-my.config.yml

time ${MARIAN_BIN} -c ${model_folder}/transformer.hi-my.config.yml 2>&1 | tee ${model_folder}/transformer.hi-my.log

Writing transformer.hi-my.sh


In [44]:
!bash transformer.hi-my.sh

[2026-10-10 12:04:08] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-10 12:04:08] [marian] Running on Thida as process 12374 with command line:
[2026-10-10 12:04:08] [marian] /home/thida/marian-dev/build/marian -c model.transformer.hi-my/transformer.hi-my.config.yml
[2026-10-10 12:04:08] [config] after: 0e
[2026-10-10 12:04:08] [config] after-batches: 0
[2026-10-10 12:04:08] [config] after-epochs: 0
[2026-10-10 12:04:08] [config] all-caps-every: 0
[2026-10-10 12:04:08] [config] allow-unk: false
[2026-10-10 12:04:08] [config] authors: false
[2026-10-10 12:04:08] [config] beam-size: 12
[2026-10-10 12:04:08] [config] bert-class-symbol: "[CLS]"
[2026-10-10 12:04:08] [config] bert-mask-symbol: "[MASK]"
[2026-10-10 12:04:08] [config] bert-masking-fraction: 0.15
[2026-10-10 12:04:08] [config] bert-sep-symbol: "[SEP]"
[2026-10-10 12:04:08] [config] bert-train-type-embeddings: true
[2026-10-10 12:04:08] [config] bert-type-vocab-size: 2
[2026-10-10 12:04:08] [config] build-

Testing

In [45]:
%%writefile evaluate_trans_hi_my.sh
#!/bin/bash

DATA="/home/thida/assignment-6/myhi_data"
MODEL_DIR="model.transformer.hi-my"
MARIAN_DEC="/home/thida/marian-dev/build/marian-decoder"

# Source က Hindi, Target က Myanmar
VOCAB="${DATA}/vocab/vocab.syl.yml"
TEST_SRC="${DATA}/test.hi"
TEST_TGT="${DATA}/syllable_segmented/syl_test.my"

> eval-result-trans-hi-my.txt

# 1. Best BLEU model
MODEL="${MODEL_DIR}/model.npz.best-bleu.npz"
HYP="${MODEL_DIR}/hyp.best-bleu.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best BLEU model  : BLEU = ${SCORE}" >> eval-result-trans-hi-my.txt

# 2. Best CE model
MODEL="${MODEL_DIR}/model.npz.best-ce-mean-words.npz"
HYP="${MODEL_DIR}/hyp.best-ce-mean-words.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Best CE model    : BLEU = ${SCORE}" >> eval-result-trans-hi-my.txt

# 3. Final model
MODEL="${MODEL_DIR}/model.npz"
HYP="${MODEL_DIR}/hyp.final.my"
${MARIAN_DEC} -m "${MODEL}" -v "${VOCAB}" "${VOCAB}" --devices 0 --output "${HYP}" < "${TEST_SRC}" 2>/dev/null
SCORE=$(cat "${HYP}" | sacrebleu "${TEST_TGT}" -b)
echo "Final model      : BLEU = ${SCORE}" >> eval-result-trans-hi-my.txt

Writing evaluate_trans_hi_my.sh


In [46]:
!bash evaluate_trans_hi_my.sh

In [47]:
!cat eval-result-trans-hi-my.txt

Best BLEU model  : BLEU = 39.2
Best CE model    : BLEU = 30.0
Final model      : BLEU = 39.2


# Result Summary

In [50]:
import subprocess
import json
from IPython.display import display, Markdown

DATA = "/home/thida/assignment-6/myhi_data"

test_hi = f"{DATA}/test.hi"
test_my = f"{DATA}/syllable_segmented/syl_test.my"

configs = [
    ("Seq2Seq", "Myanmar to Hiero", "model.s2s.syl/hyp.best-bleu.hi", test_hi),
    ("Transformer", "Myanmar to Hiero", "model.transformer.my-hi/hyp.best-bleu.hi", test_hi),
    ("Seq2Seq", "Hiero to Myanmar", "model.hi-my/hyp.best-bleu.my", test_my),
    ("Transformer", "Hiero to Myanmar", "model.transformer.hi-my/hyp.best-bleu.my", test_my)
]

md_table = "### Model Comparison (Best BLEU Scores)\n\n"
md_table += "| Architecture | Direction | BLEU Score | 1-gram | 2-gram | 3-gram | 4-gram |\n"
md_table += "|---|---|:---:|:---:|:---:|:---:|:---:|\n"

for arch, direction, hyp, ref in configs:
    sac_cmd = f"cat {hyp} | sacrebleu {ref} -f json"
    result = subprocess.run(sac_cmd, shell=True, capture_output=True, text=True)
    
    try:
        data = json.loads(result.stdout)
        bleu = round(data['score'], 2)
        ngrams = data['verbose_score'].split()[0].split('/')
        
        md_table += f"| **{arch}** | {direction} | **{bleu}** | {ngrams[0]} | {ngrams[1]} | {ngrams[2]} | {ngrams[3]} |\n"
    except Exception as e:
        md_table += f"| {arch} | {direction} | Error | - | - | - | - |\n"

display(Markdown(md_table))

### Model Comparison (Best BLEU Scores)

| Architecture | Direction | BLEU Score | 1-gram | 2-gram | 3-gram | 4-gram |
|---|---|:---:|:---:|:---:|:---:|:---:|
| **Seq2Seq** | Myanmar to Hiero | **29.5** | 54.7 | 32.3 | 23.4 | 22.0 |
| **Transformer** | Myanmar to Hiero | **22.6** | 52.5 | 29.0 | 18.2 | 11.0 |
| **Seq2Seq** | Hiero to Myanmar | **47.0** | 73.6 | 56.3 | 43.3 | 35.9 |
| **Transformer** | Hiero to Myanmar | **39.2** | 67.3 | 48.3 | 35.0 | 27.8 |
